# CTR / Engagement Opportunity Scoring: notebook 3 (within-client scoring)

What notebooks 1 and 2 showed, in plain words:

* The pre-made flag `needs_ctr_fix` is a simple rule (position <= ~10, impressions >= ~100, CTR <= 2%), so predicting it teaches nothing.
* On clients the model had never seen, position and content tiers did **not** predict CTR better than a simple position-bucket average (impression-weighted R2 about -0.01, error only 2% lower).
* The global ranking was dominated by one client (82% of the top 1000).

So this notebook drops the machine-learning model and asks a narrower, more honest question: **inside one client, which pages get fewer clicks than that client's other pages at the same position?** Each page is compared with its own siblings (leave-one-out, with shrinkage toward the global curve), and only pages that are *statistically* low are flagged, so random noise is not mistaken for opportunity. Everything is *observed / directional*.

In [ ]:
# 1. Setup and load. Needs Colab secret HF_TOKEN.
!pip install -q datasets huggingface_hub
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

import numpy as np
import pandas as pd
from datasets import load_dataset

ds = load_dataset("FlyRank/internship-lanes", "engagement_fix")
df = ds["train"].to_pandas()
print(df.shape)
print("client_has_gsc counts:", df["client_has_gsc"].value_counts(dropna=False).to_dict())

## Data quality: clients whose clicks look missing

In [ ]:
# 2. Per-client check. A client with impressions but ~zero clicks probably has a tracking gap.
base = df[(df["impressions_30d"] > 0) & df["avg_position_30d"].notna()]
pc = base.groupby("client_hash_id").agg(pages=("clicks_30d", "size"),
                                          impressions=("impressions_30d", "sum"),
                                          clicks=("clicks_30d", "sum"),
                                          has_gsc=("client_has_gsc", "max"))
pc["pooled_ctr"] = pc["clicks"] / pc["impressions"]
suspect = pc[(pc["clicks"] == 0) | (pc["pooled_ctr"] < 0.0005)]
print(f"Clients: {len(pc)} | suspect (almost no clicks): {len(suspect)}")
print(f"Suspect clients hold {suspect['impressions'].sum() / pc['impressions'].sum():.1%} of impressions and {suspect['pages'].sum() / pc['pages'].sum():.1%} of pages")
print("has_gsc among suspect clients:", suspect["has_gsc"].value_counts(dropna=False).to_dict())
print()
print(pc.sort_values("pooled_ctr").head(8).round(5).to_string())

In [ ]:
# 3. Analysis set: drop suspect clients, keep pages with enough impressions
MIN_IMP = 100
good_clients = set(pc.index) - set(suspect.index)
d = base[base["client_hash_id"].isin(good_clients) & (base["impressions_30d"] >= MIN_IMP)].copy()
d["ctr"] = d["clicks_30d"] / d["impressions_30d"]
print(f"Kept {len(d):,} pages from {d['client_hash_id'].nunique()} clients")
print(f"Biggest client: {d.groupby('client_hash_id').size().max() / len(d):.1%} of pages, "
      f"{d.groupby('client_hash_id')['impressions_30d'].sum().max() / d['impressions_30d'].sum():.1%} of impressions")

## The expectation: a page vs its own client

For each page, take its client's pooled CTR in the same position bucket **leaving the page itself out**, then blend it with the global curve. A client with little data in that bucket leans on the global curve (`K` controls how fast). No model is fitted, so there is nothing to over-fit.

In [ ]:
# 4. Leave-one-out, shrunk, within-client expectation
bins = [0, 1.5, 2.5, 3.5, 4.5, 5.5, 7.5, 10.5, 15.5, 20.5, 1000]
d["bucket"] = pd.cut(d["avg_position_30d"], bins=bins, labels=False)
K = 5000   # impressions of "evidence" needed before trusting the client's own curve about as much as the global one

glob = d.groupby("bucket")[["clicks_30d", "impressions_30d"]].sum()
glob_ctr = glob["clicks_30d"] / glob["impressions_30d"]
d["global_ctr"] = d["bucket"].map(glob_ctr)

cb = d.groupby(["client_hash_id", "bucket"])[["clicks_30d", "impressions_30d"]].transform("sum")
loo_imp = cb["impressions_30d"] - d["impressions_30d"]
loo_clk = cb["clicks_30d"] - d["clicks_30d"]
loo_ctr = np.where(loo_imp > 0, loo_clk / loo_imp.where(loo_imp > 0, 1), 0.0)
wgt = loo_imp / (loo_imp + K)
d["expected_ctr"] = np.clip(wgt * loo_ctr + (1 - wgt) * d["global_ctr"], 1e-6, 1)
d["evidence_weight"] = wgt
print("Median share of expectation that comes from the client's own pages:", round(float(wgt.median()), 3))

In [ ]:
# 5. Honest check: does the within-client expectation predict a page better than the global curve?
from sklearn.metrics import mean_absolute_error, r2_score
yt, wt = d["ctr"], d["impressions_30d"]
mae_g = mean_absolute_error(yt, d["global_ctr"], sample_weight=wt); r2_g = r2_score(yt, d["global_ctr"], sample_weight=wt)
mae_c = mean_absolute_error(yt, d["expected_ctr"], sample_weight=wt); r2_c = r2_score(yt, d["expected_ctr"], sample_weight=wt)
print(f"Global position curve     : impression-weighted MAE {mae_g:.5f} | weighted R2 {r2_g:.3f}")
print(f"Within-client (LOO, shrunk): impression-weighted MAE {mae_c:.5f} | weighted R2 {r2_c:.3f}")
print(f"MAE change: {(mae_c - mae_g) / mae_g:+.1%}")
print("Note: this tests predicting a page from its sibling pages in the SAME client, which is the question this score answers.")

## Only flag what is statistically low

With few clicks, a low CTR can be luck. For each page, ask: if its true CTR were really the expected CTR, how likely is it to see this few clicks or fewer? A small probability (`p < 0.01`) means the shortfall is unlikely to be noise. This is a screening aid, not proof.

In [ ]:
# 6. Statistical filter and missed clicks
from scipy.stats import poisson
d["exp_clicks"] = d["expected_ctr"] * d["impressions_30d"]
d["p_low"] = poisson.cdf(d["clicks_30d"], d["exp_clicks"])
d["missed_clicks"] = (d["exp_clicks"] - d["clicks_30d"]).clip(lower=0).round(1)
flag = d[(d["p_low"] < 0.01) & (d["missed_clicks"] >= 5)].copy()
print(f"Flagged as statistically low: {len(flag):,} of {len(d):,} pages ({len(flag)/len(d):.1%})")
print(f"Their missed clicks: {flag['missed_clicks'].sum():,.0f} of {d['missed_clicks'].sum():,.0f} total estimated missed clicks")

cc = flag["client_hash_id"].value_counts()
print(f"Flagged pages come from {flag['client_hash_id'].nunique()} clients; largest client = {cc.iloc[0] / len(flag):.1%}")

## Ranked recommendations

Each flagged page gets a reason and a suggested *review* action (directional, not proof that a change will work). Ranking happens **inside each client first** (top 5 per client), then a final list is built, so one big client cannot swamp everyone else.

In [ ]:
# 7. Reason codes, action playbook, top-5-per-client list
def action(r):
    p = r["avg_position_30d"]
    if p <= 3.5:
        return "Check SERP layout and query mix first (top-3 page with low CTR), then review title/meta"
    if p <= 10.5:
        return "Review title and meta description (striking distance, CTR below client norm)"
    return "Monitor (low position, CTR gap may be small in absolute terms)"

def reasons(r):
    out = ["CTR below client's own pattern at this position"]
    if r["clicks_30d"] < 0.5 * r["exp_clicks"]: out.append("under half of expected clicks")
    if r["impressions_30d"] >= d["impressions_30d"].quantile(0.75): out.append("high visibility")
    return "; ".join(out)

flag["reason_codes"] = flag.apply(reasons, axis=1)
flag["suggested_review"] = flag.apply(action, axis=1)
flag["rank_in_client"] = flag.groupby("client_hash_id")["missed_clicks"].rank(ascending=False, method="first")
playbook = flag[flag["rank_in_client"] <= 5].sort_values("missed_clicks", ascending=False)

print(f"Playbook (top 5 per client): {len(playbook)} pages from {playbook['client_hash_id'].nunique()} clients")
print(playbook["suggested_review"].value_counts().to_string())
print()
show = ["content_hash_id", "avg_position_30d", "impressions_30d", "ctr", "expected_ctr", "missed_clicks", "p_low", "suggested_review"]
print("Top 10 (local use only, do NOT publish IDs):")
print(playbook[show].head(10).round(4).to_string(index=False))

In [ ]:
# 8. Does the pre-made flag agree? (sanity check only, it is a rule on CTR/position so some overlap is expected)
print(f"needs_ctr_fix rate among flagged pages: {flag['needs_ctr_fix'].mean():.3f} vs {d['needs_ctr_fix'].mean():.3f} across all kept pages")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(d["exp_clicks"].clip(lower=1), d["clicks_30d"].clip(lower=1), s=3, alpha=0.15, label="pages")
ax.scatter(flag["exp_clicks"].clip(lower=1), flag["clicks_30d"].clip(lower=1), s=6, alpha=0.6, label="flagged (p < 0.01)")
mx = max(d["exp_clicks"].max(), d["clicks_30d"].max())
ax.plot([1, mx], [1, mx], linewidth=1)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Expected clicks (within-client)"); ax.set_ylabel("Actual clicks"); ax.set_title("Actual vs expected clicks")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# 9. Numbers to copy back to me (no IDs, no client info)
print("===== COPY EVERYTHING BELOW THIS LINE AND SEND IT TO ME =====")
print(f"Clients total {len(pc)} | suspect (no clicks) {len(suspect)} ({suspect['impressions'].sum() / pc['impressions'].sum():.1%} of impressions)")
print(f"has_gsc among suspect: {suspect['has_gsc'].value_counts(dropna=False).to_dict()}")
print(f"Kept {len(d)} pages, {d['client_hash_id'].nunique()} clients")
print(f"Global curve: MAE {mae_g:.5f} R2 {r2_g:.3f} | Within-client: MAE {mae_c:.5f} R2 {r2_c:.3f} | MAE change {(mae_c - mae_g) / mae_g:+.1%}")
print(f"Flagged {len(flag)} ({len(flag)/len(d):.1%}); share of missed clicks {flag['missed_clicks'].sum() / d['missed_clicks'].sum():.1%}")
print(f"Flagged from {flag['client_hash_id'].nunique()} clients; largest {cc.iloc[0] / len(flag):.1%}")
print(f"Playbook {len(playbook)} pages from {playbook['client_hash_id'].nunique()} clients")
print(playbook["suggested_review"].value_counts().to_string())
print(f"needs_ctr_fix among flagged {flag['needs_ctr_fix'].mean():.3f} vs {d['needs_ctr_fix'].mean():.3f} overall")